In [17]:
from ultralytics import YOLO
import cv2
import numpy as np
import os

# ── CONFIG ──────────────────────────────────────
VIDEO_PATH  = "traffic3.mp4"
MODEL_PATH  = "yolo26n.pt"    
OUTPUT_PATH = "output/stage1_2_output.mp4"

ROAD_USER_CLASSES = [0, 1, 2, 3, 5, 7]
SKIP_FRAMES = 2  
CLASS_NAMES = {
    0: "Person", 1: "Bicycle", 2: "Car",
    3: "Motorcycle", 5: "Bus", 7: "Truck"
}

CLASS_COLORS = {
    0: (0, 255, 0),    
    1: (255, 165, 0),  
    2: (0, 200, 255),  
    3: (255, 0, 255),  
    5: (0, 0, 255),    
    7: (255, 255, 0),  
}

TRACK_HISTORY_LEN = 30

os.makedirs("output", exist_ok=True)

In [18]:
model = YOLO(MODEL_PATH)

cap = cv2.VideoCapture(VIDEO_PATH)
W   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
FPS = cap.get(cv2.CAP_PROP_FPS)
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Video: {W}x{H} @ {FPS:.1f} FPS | {TOTAL_FRAMES} frames")

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out    = cv2.VideoWriter(OUTPUT_PATH, fourcc, FPS, (W, H))

track_history = {}
frame_idx = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_idx += 1

    
    if frame_idx % SKIP_FRAMES != 0:
        out.write(frame)   
        continue

    
    small = cv2.resize(frame, (320, 180))

    results = model.track(
        small,
        persist=True,
        tracker="bytetrack.yaml",
        conf=0.4,
        classes=ROAD_USER_CLASSES,
        verbose=False,
        imgsz=320,   
        half=True,    
    )

    
    scale_x = W / 320
    scale_y = H / 180

    for r in results:
        if r.boxes is None:
            continue
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            
            x1 = int(x1 * scale_x); x2 = int(x2 * scale_x)
            y1 = int(y1 * scale_y); y2 = int(y2 * scale_y)
            cx, cy = (x1+x2)//2, (y1+y2)//2

            cls_id   = int(box.cls[0])
            conf     = float(box.conf[0])
            cls_name = CLASS_NAMES.get(cls_id, "Unknown")
            color    = CLASS_COLORS.get(cls_id, (255,255,255))
            track_id = int(box.id[0]) if box.id is not None else -1

            if track_id != -1:
                if track_id not in track_history:
                    track_history[track_id] = []
                track_history[track_id].append((cx, cy))
                if len(track_history[track_id]) > 30:
                    track_history[track_id].pop(0)

            cv2.rectangle(frame, (x1,y1), (x2,y2), color, 2)
            label = f"{cls_name} #{track_id} {conf:.2f}"
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
            cv2.rectangle(frame, (x1, y1-th-6), (x1+tw+4, y1), color, -1)
            cv2.putText(frame, label, (x1+2, y1-4),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,0,0), 1)

            if track_id != -1 and track_id in track_history:
                pts = track_history[track_id]
                for i in range(1, len(pts)):
                    cv2.line(frame, pts[i-1], pts[i], color, 1)

    if frame_idx % 200 == 0:
        print(f"⏳ Frame {frame_idx}/{TOTAL_FRAMES} done...")

    out.write(frame)

cap.release()
out.release()
print(f"Done! Saved to: {OUTPUT_PATH}")

Video: 640x360 @ 30.0 FPS | 312 frames
⏳ Frame 200/312 done...
Done! Saved to: output/stage1_2_output.mp4


In [3]:
def get_nearest_edge_distance(b1, b2):
    """Distance between nearest edges of two bounding boxes."""
    x1_1, y1_1, x2_1, y2_1 = b1
    x1_2, y1_2, x2_2, y2_2 = b2

    dx = max(0, max(x1_1, x1_2) - min(x2_1, x2_2))
    dy = max(0, max(y1_1, y1_2) - min(y2_1, y2_2))
    return np.sqrt(dx**2 + dy**2)

def get_box_size(box):
    """Average of width and height — used as scale factor."""
    x1, y1, x2, y2 = box
    return ((x2 - x1) + (y2 - y1)) / 2

def are_too_close(box1, box2, scale_factor=2.0):
    """
    Two objects are 'too close' if their edge distance
    is less than scale_factor * average object size.
    This handles camera depth — far objects are smaller
    so threshold shrinks automatically.
    """
    dist  = get_nearest_edge_distance(box1, box2)
    size1 = get_box_size(box1)
    size2 = get_box_size(box2)
    avg_size  = (size1 + size2) / 2
    threshold = scale_factor * avg_size
    return dist < threshold, dist, threshold

VEHICLE_CLASSES = {2, 3, 5, 7}   

def is_vehicle(cls_id):
    return cls_id in VEHICLE_CLASSES

✅ Stage 3 proximity functions ready!


In [4]:
def get_velocity(track_id, track_history):
    """Estimate velocity from last 5 positions."""
    if track_id not in track_history:
        return (0, 0)
    pts = track_history[track_id]
    if len(pts) < 2:
        return (0, 0)
    
    n = min(5, len(pts))
    vx = (pts[-1][0] - pts[-n][0]) / n
    vy = (pts[-1][1] - pts[-n][1]) / n
    return (vx, vy)

def get_relative_velocity(id1, id2, track_history):
    """
    Dot product of velocities — negative means
    they are moving TOWARD each other (higher risk).
    """
    v1 = get_velocity(id1, track_history)
    v2 = get_velocity(id2, track_history)
    
    dot = v1[0]*v2[0] + v1[1]*v2[1]
    speed1 = np.sqrt(v1[0]**2 + v1[1]**2)
    speed2 = np.sqrt(v2[0]**2 + v2[1]**2)
    return dot, speed1, speed2

def compute_risk_score(box1, box2, cls1, cls2, id1, id2, track_history):
    """
    Risk Score 0–100:
      40% — proximity (how close are they?)
      30% — object type combo (pedestrian involved = higher risk)
      30% — relative velocity (moving toward each other?)
    """
    dist  = get_nearest_edge_distance(box1, box2)
    size1 = get_box_size(box1)
    size2 = get_box_size(box2)
    avg_size  = (size1 + size2) / 2

    
    dist_ratio   = max(0, 1 - dist / (3.0 * avg_size))
    dist_score   = dist_ratio * 40

    
    person_involved = (cls1 == 0 or cls2 == 0)
    both_vehicles   = is_vehicle(cls1) and is_vehicle(cls2)
    if person_involved:
        type_score = 30     
    elif both_vehicles:
        type_score = 20     
    else:
        type_score = 10     

    
    dot, speed1, speed2 = get_relative_velocity(id1, id2, track_history)
    total_speed  = speed1 + speed2
    
    if total_speed > 0:
        approach_factor = max(0, -dot / (total_speed + 1e-5))
    else:
        approach_factor = 0
    vel_score = approach_factor * 30

    total_score = dist_score + type_score + vel_score

    # ── Risk Level
    if total_score >= 65:
        level = "HIGH"
        color = (0, 0, 255)     
    elif total_score >= 35:
        level = "MEDIUM"
        color = (0, 165, 255)    
    else:
        level = "LOW"
        color = (0, 255, 255)   

    return round(total_score, 1), level, color

✅ Stage 4 risk scoring functions ready!


In [5]:
HIGH_RISK_FRAMES_NEEDED = 7      
COOLDOWN_FRAMES         = 60     

pair_high_risk_count = {}
pair_cooldown        = {}
near_miss_log        = []
near_miss_count      = 0

def get_pair_key(id1, id2):
    return (min(id1, id2), max(id1, id2))

def update_near_miss(pair_key, risk_level, frame_idx):
    global near_miss_count

    if pair_key in pair_cooldown:
        pair_cooldown[pair_key] -= 1
        if pair_cooldown[pair_key] <= 0:
            del pair_cooldown[pair_key]
        return False

    if risk_level == "HIGH":
        pair_high_risk_count[pair_key] = pair_high_risk_count.get(pair_key, 0) + 1
    else:
        pair_high_risk_count[pair_key] = 0

    if pair_high_risk_count.get(pair_key, 0) >= HIGH_RISK_FRAMES_NEEDED:
        near_miss_count += 1
        pair_high_risk_count[pair_key] = 0
        pair_cooldown[pair_key] = COOLDOWN_FRAMES
        near_miss_log.append({
            "event": near_miss_count,
            "pair": pair_key,
            "frame": frame_idx
        })
        return True

    return False

✅ Balanced thresholds — 7 frames needed, 60 cooldown!


In [19]:
OUTPUT_FULL = "output/full_pipeline_output_v4.mp4"


active_near_miss_display = {}   
DISPLAY_DURATION         = 45   

pair_high_risk_count = {}
pair_cooldown        = {}
near_miss_log        = []
near_miss_count      = 0
track_history        = {}
seen_ids             = {}
active_ids           = {}
exited_ids           = set()
class_entered        = {0:0, 1:0, 2:0, 3:0, 5:0, 7:0}
class_exited         = {0:0, 1:0, 2:0, 3:0, 5:0, 7:0}

HIGH_RISK_FRAMES_NEEDED = 7
COOLDOWN_FRAMES         = 60
PROXIMITY_SCALE         = 1.5

model = YOLO(MODEL_PATH)
cap   = cv2.VideoCapture(VIDEO_PATH)
W     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
FPS   = cap.get(cv2.CAP_PROP_FPS)
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Video: {W}x{H} @ {FPS:.1f} FPS | {TOTAL_FRAMES} frames")

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out    = cv2.VideoWriter(OUTPUT_FULL, fourcc, FPS, (W, H))

frame_idx        = 0
flash_counter    = 0
current_risk_lvl = "LOW"
last_detections  = []


def draw_main_dashboard(frame, near_miss_count, current_risk, W, H):
    PANEL_W = 200
    x0 = W - PANEL_W

    overlay = frame.copy()
    cv2.rectangle(overlay, (x0, 0), (W, 115), (15,15,15), -1)
    cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)

    cv2.putText(frame, "TRAFFIC MONITOR", (x0+8, 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.48, (0,210,255), 2)
    cv2.line(frame, (x0+5, 27), (W-5, 27), (80,80,80), 1)

    cv2.putText(frame, "NEAR-MISSES", (x0+8, 48),
                cv2.FONT_HERSHEY_SIMPLEX, 0.42, (180,180,180), 1)
    nm_color = (0,0,255) if near_miss_count > 0 else (0,255,100)
    cv2.putText(frame, str(near_miss_count), (x0+8, 88),
                cv2.FONT_HERSHEY_DUPLEX, 1.6, nm_color, 3)

    risk_colors = {"LOW":(0,255,100), "MEDIUM":(0,165,255), "HIGH":(0,0,255)}
    r_col = risk_colors.get(current_risk, (255,255,255))
    cv2.putText(frame, f"RISK: {current_risk}", (x0+95, 72),
                cv2.FONT_HERSHEY_SIMPLEX, 0.52, r_col, 2)
    cv2.line(frame, (x0+5, 100), (W-5, 100), (80,80,80), 1)


def update_near_miss_strict(pair_key, risk_level, frame_idx):
    global near_miss_count

    if pair_key in pair_cooldown:
        pair_cooldown[pair_key] -= 1
        if pair_cooldown[pair_key] <= 0:
            del pair_cooldown[pair_key]
        return False

    if risk_level == "HIGH":
        pair_high_risk_count[pair_key] = pair_high_risk_count.get(pair_key, 0) + 1
    else:
        pair_high_risk_count[pair_key] = 0

    if pair_high_risk_count.get(pair_key, 0) >= HIGH_RISK_FRAMES_NEEDED:
        near_miss_count += 1
        pair_high_risk_count[pair_key] = 0
        pair_cooldown[pair_key] = COOLDOWN_FRAMES
        near_miss_log.append({
            "event": near_miss_count,
            "pair": pair_key,
            "frame": frame_idx
        })
        return True

    return False

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_idx += 1

    run_yolo = (frame_idx % 2 == 0)

    if run_yolo:
        results = model.track(
            frame,
            persist=True,
            tracker="bytetrack.yaml",
            conf=0.25,
            classes=ROAD_USER_CLASSES,
            verbose=False,
            imgsz=640,
            half=True,
        )

        current_frame_ids = {}
        last_detections   = []

        for r in results:
            if r.boxes is None:
                continue
            for box in r.boxes:
                x1,y1,x2,y2 = map(int, box.xyxy[0])
                cx,cy = (x1+x2)//2, (y1+y2)//2
                cls_id   = int(box.cls[0])
                conf     = float(box.conf[0])
                track_id = int(box.id[0]) if box.id is not None else -1

                if track_id != -1:
                    if track_id not in track_history:
                        track_history[track_id] = []
                    track_history[track_id].append((cx,cy))
                    if len(track_history[track_id]) > 30:
                        track_history[track_id].pop(0)

                    current_frame_ids[track_id] = cls_id

                    if track_id not in seen_ids:
                        seen_ids[track_id] = cls_id
                        class_entered[cls_id] += 1

                last_detections.append({
                    "box":(x1,y1,x2,y2), "cls":cls_id,
                    "conf":conf, "id":track_id, "center":(cx,cy)
                })

        for tid, cls_id in list(active_ids.items()):
            if tid not in current_frame_ids and tid not in exited_ids:
                class_exited[cls_id] += 1
                exited_ids.add(tid)

        active_ids = current_frame_ids

    detections = last_detections
    
    near_miss_ids  = set()
    pair_risk_info = {}
    frame_max_risk = "LOW"

    for i in range(len(detections)):
        for j in range(i+1, len(detections)):
            d1, d2 = detections[i], detections[j]
            if d1["id"] == -1 or d2["id"] == -1:
                continue
            if not (is_vehicle(d1["cls"]) or is_vehicle(d2["cls"])):
                continue

            too_close, dist, thresh = are_too_close(
                d1["box"], d2["box"], scale_factor=PROXIMITY_SCALE
            )
            if not too_close:
                continue

            score, level, risk_color = compute_risk_score(
                d1["box"], d2["box"],
                d1["cls"], d2["cls"],
                d1["id"],  d2["id"],
                track_history
            )

            if level == "HIGH":
                frame_max_risk = "HIGH"
            elif level == "MEDIUM" and frame_max_risk != "HIGH":
                frame_max_risk = "MEDIUM"

            pair_key     = get_pair_key(d1["id"], d2["id"])
            is_near_miss = update_near_miss_strict(pair_key, level, frame_idx)

            pair_risk_info[pair_key] = (
                score, level, risk_color, d1["center"], d2["center"]
            )
            if is_near_miss:
                near_miss_ids.add(d1["id"])
                near_miss_ids.add(d2["id"])

    current_risk_lvl = frame_max_risk

    for tid in near_miss_ids:
        active_near_miss_display[tid] = DISPLAY_DURATION

    for tid in list(active_near_miss_display.keys()):
        active_near_miss_display[tid] -= 1
        if active_near_miss_display[tid] <= 0:
            del active_near_miss_display[tid]

    flash_counter += 1
    for d in detections:
        tid = d["id"]
        if tid not in active_near_miss_display:
            continue   

        x1,y1,x2,y2 = d["box"]
        cls_name = CLASS_NAMES.get(d["cls"], "?")

        
        if flash_counter % 6 < 3:
            box_color  = (0, 0, 255)      
            text_color = (255, 255, 255)  
        else:
            box_color  = (0, 0, 180)      
            text_color = (220, 220, 220)

        # Thick box
        cv2.rectangle(frame, (x1,y1), (x2,y2), box_color, 3)

        # Label
        label = f"⚠ {cls_name} #{tid}"
        (lw,lh),_ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
        cv2.rectangle(frame, (x1, y1-lh-7), (x1+lw+6, y1), box_color, -1)
        cv2.putText(frame, label, (x1+3, y1-4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, text_color, 1)

       
        remaining = active_near_miss_display.get(tid, 0)
        bar_len   = int((remaining / DISPLAY_DURATION) * (x2 - x1))
        cv2.rectangle(frame, (x1, y2+2), (x2,    y2+6), (60,60,60),  -1)
        cv2.rectangle(frame, (x1, y2+2), (x1+bar_len, y2+6), box_color, -1)

   
    for pair_key, (score, level, risk_color, c1, c2) in pair_risk_info.items():
        id1, id2 = pair_key
        if id1 not in active_near_miss_display and id2 not in active_near_miss_display:
            continue
        cv2.line(frame, c1, c2, (0,0,255), 2)
        mid   = ((c1[0]+c2[0])//2, (c1[1]+c2[1])//2)
        badge = f"HIGH  {score:.0f}/100"
        (bw,bh),_ = cv2.getTextSize(badge, cv2.FONT_HERSHEY_SIMPLEX, 0.48, 1)
        cv2.rectangle(frame, (mid[0]-4, mid[1]-bh-5),
                      (mid[0]+bw+4, mid[1]+2), (20,20,20), -1)
        cv2.rectangle(frame, (mid[0]-4, mid[1]-bh-5),
                      (mid[0]+bw+4, mid[1]+2), (0,0,255), 1)
        cv2.putText(frame, badge, (mid[0], mid[1]-2),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.48, (0,0,255), 1)

    
    if active_near_miss_display and (flash_counter % 10 < 5):
        cv2.rectangle(frame, (0, H-50), (W-200, H), (0,0,160), -1)
        cv2.putText(frame, "⚠  NEAR-MISS DETECTED  ⚠",
                    (20, H-14),
                    cv2.FONT_HERSHEY_DUPLEX, 0.8, (255,255,255), 2)

    
    draw_main_dashboard(frame, near_miss_count, current_risk_lvl, W, H)

    out.write(frame)

    if frame_idx % 300 == 0:
        print(f"{frame_idx}/{TOTAL_FRAMES} | "
              f"Risk: {current_risk_lvl} | "
              f"Near-misses: {near_miss_count}")

cap.release()
out.release()

print(f"\nDone! Saved: {OUTPUT_FULL}")
print(f"Total near-miss events: {near_miss_count}")
print(f"\nNear-Miss Log:")
for e in near_miss_log:
    ts = round(e['frame'] / FPS, 1)
    print(f"   Event #{e['event']} | Pair {e['pair']} | Frame {e['frame']} | {ts}s")

Video: 640x360 @ 30.0 FPS | 312 frames
300/312 | Risk: HIGH | Near-misses: 8

Done! Saved: output/full_pipeline_output_v4.mp4
Total near-miss events: 8

Near-Miss Log:
   Event #1 | Pair (2, 95) | Frame 130 | 4.3s
   Event #2 | Pair (87, 95) | Frame 130 | 4.3s
   Event #3 | Pair (95, 98) | Frame 132 | 4.4s
   Event #4 | Pair (2, 106) | Frame 142 | 4.7s
   Event #5 | Pair (98, 106) | Frame 142 | 4.7s
   Event #6 | Pair (87, 106) | Frame 146 | 4.9s
   Event #7 | Pair (106, 119) | Frame 180 | 6.0s
   Event #8 | Pair (98, 183) | Frame 300 | 10.0s
